<a href="https://colab.research.google.com/github/nam5588/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
print("Menu Detector")

Menu Detector


In [52]:
# ------------------
# Import Libraries
# ------------------
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np


In [53]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("Dataset Path: ", DATASET_PATH)
# ^  Datasetni pathni oldik

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # Label Grouping | Class Consolidation
    'cheesecake': 'dessert',     # Label Grouping | Class Consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}
# ^  Labellarni berdik yani "Nomladik"


CLASSES = ['hamburger', 'hot_dog', 'dessert','kebab', 'pilaf']
# ^  Classlarga bo'ldik -  [ 1  2  3  4  5 ]
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
# ^  Indexladik 5 ta classimizni
NUM_CLASSES = len(CLASSES)
# ^  Classlar uzunligini oldik

print(NUM_CLASSES)
print(CLASS_TO_IDX)


transform = transforms.Compose([      # pipeline kabi ketmaketlik-orderni ushlaydi
    transforms.Resize((224, 224)),    # resize qilib fine-tuning uchun moslash
    transforms.ToTensor(),            # tensorga (raqamlarga olamiz = 0.0 ~ 0.1)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  # resize+tensorni normallaydi
])

Dataset Path:  /content/drive/MyDrive/food101_dataset
5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [55]:
# ---------------------
# Custtom Dataset Class
# ---------------------

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    # print("images_length:", len(self.images))
    return len(self.images)

  def __getitem__(self, idx):
    img_path = self.images[idx]
    # print("img_path:", img_path)
    label = self.labels[idx]
    # print("label:", label)
    try:
      image = Image.open(img_path)
      if image.mode == "P" or image.mode =='RGBA':
        image = image.convert('RGBA').convert('RGB')
      else:
        image = image.convert('RGB')
    except (UnidentifiedImageError, OSError):
      print('Skipping broken image:', img_path)
      return self.__getitem__((idx + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label


In [56]:
# ---------------------
# Gather an Split Data
# ---------------------

all_images = []
for original_class, mapped_classs in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class)
  print('class_path:', class_path)
  if not os.path.exists(class_path):
    print(f"Warning: {original_class} not found")
    continue
  for img in os.listdir(class_path):
    if img.endswith(('.jpg', '.png', '.jpeg')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASS_TO_IDX[mapped_classs]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
3333


In [57]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [58]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [59]:
# pretrained model
model = mobilenet_v2(weights='IMAGENET1K_V1') # pretrained model | light weight | CNN
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone | model layer freeze

In [60]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
mdoel = model.to(device)

cuda


In [61]:
criterion = nn.CrossEntropyLoss() # Loss Function - plov = '70%', burger = '30%'
optimizer = optim.Adam(model.parameters(), lr=0.001) # weight
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | speed 10%-20%

In [62]:
# ---------------------
# Training Loop
# ---------------------

NUM_EPOCHS = 10       # toliq necha marta test qilishimiz | aylanishi
best_accuracy = 0.0   # yaxshi natija chiqib uni save qilganda | keyin ishlarish u-n

for epoch in range(NUM_EPOCHS):
  model.train()
  running_loss = 0.0      # train paytidagi yoqotishni aniqlash 70% | 30% = loss = 30% ga

  for images, labels in train_loader: # Forward va Backward - Asosiy qism Oldinga borib oqish va qayta oqish uchun orqaga qaytish
    images, labels = images.to(device), labels.to(device) # Rasm va Labellarni GPUga otkazdik
    optimizer.zero_grad()             # har safargi aylanganda backprop. dan avval optimizerni (zero the grad) = 0 ga tenglanadi
    outputs = model(images)           # Forward pass | 5 classes -> cnn orqali featurelarni ajatib tanish
    loss = criterion(outputs, labels) # arg (1 ni , 2 dan) qanchalik faqrli ekanin lossni (xatosini) aniqlash
    loss.backward()                   # ortga qaytarish yangi epoch
    optimizer.step()                  # weightlarni aniqlab yangilar | Adam optimizer
    running_loss += loss.item()       # run bolish paytida losslarni hisoblab boradi

# Validation              Loop 1 - yakunlanib o'rganish jarayonini ko'ramiz baholaymiz
  model.eval()            # modelni baholash
  correct = 0             # nechta to'g'ri test qilindi
  total = 0               # nechta test qilindi
  with torch.no_grad():
    for images, labels in val_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  val_acc = 100 * correct / total   # Validationdan qanchalik accuracy (toza) ishladi
  print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%')

  if val_acc > best_accuracy:   # yaxshiroq accuracy chiqsa uni doimiy saqlab boramiz!
    best_accuracy = val_acc
    torch.save(model.state_dict(), '/content/menu_detector.pth')
    print('Saved new best model!')


Epoch [1/10], Loss: 0.5399, Val Accuracy: 85.37%
Saved new best model!
Epoch [2/10], Loss: 0.2982, Val Accuracy: 82.85%
Epoch [3/10], Loss: 0.2986, Val Accuracy: 86.21%
Saved new best model!
Epoch [4/10], Loss: 0.2278, Val Accuracy: 87.89%
Saved new best model!
Epoch [5/10], Loss: 0.1975, Val Accuracy: 84.89%
Epoch [6/10], Loss: 0.3174, Val Accuracy: 87.65%
Epoch [7/10], Loss: 0.2378, Val Accuracy: 90.29%
Saved new best model!
Epoch [8/10], Loss: 0.1266, Val Accuracy: 82.97%
Epoch [9/10], Loss: 0.1209, Val Accuracy: 87.53%
Epoch [10/10], Loss: 0.1216, Val Accuracy: 88.25%
